In [ ]:
# =============================================================================
# AVALIAÇÃO FINAL DOS MODELOS TREINADOS
# =============================================================================
# Este script carrega os 5 modelos "from scratch" salvos (best_*.pth),
# avalia-os usando o conjunto de validação e armazena os resultados em um arquivo JSON
# e gera um gráfico comparativo salvo em PNG.
#
# Certifique-se de que:
# - As funções e classes do seu módulo "your_dataset_module" estão importadas corretamente.
# - Os arquivos de pesos ("best_multiscale.pth", etc.) estejam disponíveis.
# - Os caminhos dos diretórios ("images/val", "labels/val") e dos mapeamentos (helpers/*.json)
#   estejam corretos para o seu ambiente.
# =============================================================================

import os
import json
import torch
import random
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Subset, DataLoader

from modelos_scratch import MultiScaleCNN, TinyYOLOStyle, SharedTrunkMultiTask, CNNWithAttention, PyramidCNN
from dataset_class import MultiTaskObjectDetectionDataset, collate_fn

# =============================================================================
# Configuração do dataset e dos parâmetros de validação
# =============================================================================
transform = transforms.Compose([transforms.ToTensor()])

# Carrega o dataset de validação; aqui usamos 50% dos dados para avaliação
val_ds = MultiTaskObjectDetectionDataset("images/val", "labels/val", transform)
subset_size = int(0.5 * len(val_ds))
val_subset = Subset(val_ds, random.sample(range(len(val_ds)), subset_size))
val_loader = DataLoader(val_subset, batch_size=8, collate_fn=collate_fn, num_workers=0)

# Definir dispositivo (GPU se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carregar os mapeamentos de categorias, weather, scene e timeofday
with open("helpers/categories.json", "r") as f:
    category_to_label = json.load(f)
with open("helpers/weather.json", "r") as f:
    weather_to_label = json.load(f)
with open("helpers/scene.json", "r") as f:
    scene_to_label = json.load(f)
with open("helpers/timeofday.json", "r") as f:
    timeofday_to_label = json.load(f)

num_classes = len(category_to_label) + 1  # +1 para background, se aplicável
num_weather = len(weather_to_label)
num_scene = len(scene_to_label)
num_time = len(timeofday_to_label)

# =============================================================================
# Definições de funções auxiliares
# =============================================================================
def compute_iou(box1, box2):
    """
    Calcula o IoU entre duas caixas [x1, y1, x2, y2].
    """
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def evaluate_model(model, loader, device, confidence_threshold=0.5, iou_threshold=0.5):
    """
    Avalia o modelo no conjunto de validação.
    
    Para modelos que geram atributos (SharedTrunkMultiTask) avalia acurácia de weather, scene e timeofday.
    Também avalia detecção (comparando detecções com as ground-truth a partir de IoU e classificação).
    Retorna um dicionário com métricas.
    """
    model.eval()
    total_attr, correct_weather, correct_scene, correct_time = 0, 0, 0, 0
    total_gt, correct_det, total_detections = 0, 0, 0

    with torch.no_grad():
        for images, targets, attrs in loader:
            if not images:
                continue
            images = [img.to(device) for img in images]
            # Se o modelo for SharedTrunkMultiTask, o forward retorna uma tupla: (detecções, dict de atributos)
            if isinstance(model, SharedTrunkMultiTask):
                det_output, attr_output = model(images, None, None)
                w_logits = attr_output["weather"]
                s_logits = attr_output["scene"]
                t_logits = attr_output["timeofday"]
                
                # Para avaliação dos atributos, assumindo que cada imagem possui um atributo global
                attr_preds_w = torch.argmax(w_logits, dim=1).cpu()
                attr_preds_s = torch.argmax(s_logits, dim=1).cpu()
                attr_preds_t = torch.argmax(t_logits, dim=1).cpu()
                attr_true = [a["weather"] for a in attrs]
                attr_true_s = [a["scene"] for a in attrs]
                attr_true_t = [a["timeofday"] for a in attrs]
                correct_weather += (attr_preds_w == torch.tensor(attr_true)).sum().item()
                correct_scene   += (attr_preds_s == torch.tensor(attr_true_s)).sum().item()
                correct_time    += (attr_preds_t == torch.tensor(attr_true_t)).sum().item()
                total_attr += len(attrs)
                dets = det_output
            else:
                dets = model(images, None)

            # Avaliação de detecção para cada imagem
            for i in range(len(images)):
                gt_boxes = targets[i]["boxes"].cpu().numpy()
                gt_labels = targets[i]["labels"].cpu().numpy()
                total_gt += len(gt_boxes)

                # Para os modelos sem score, assumimos score 1
                pred_boxes = dets[i]["boxes"].cpu().numpy()
                pred_labels = dets[i]["labels"].cpu().numpy()
                pred_scores = dets[i].get("scores", torch.ones_like(dets[i]["labels"])).cpu().numpy()

                keep = pred_scores >= confidence_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                total_detections += len(pred_boxes)
                for j, gt in enumerate(gt_boxes):
                    best_iou, best_idx = 0, -1
                    for k, pred in enumerate(pred_boxes):
                        iou = compute_iou(gt, pred)
                        if iou > best_iou:
                            best_iou, best_idx = iou, k
                    if best_iou >= iou_threshold and pred_labels[best_idx] == gt_labels[j]:
                        correct_det += 1

    metrics = {}
    if total_attr:
        metrics["attr_acc_weather"] = correct_weather / total_attr
        metrics["attr_acc_scene"]   = correct_scene   / total_attr
        metrics["attr_acc_time"]    = correct_time    / total_attr
    else:
        metrics["attr_acc_weather"] = metrics["attr_acc_scene"] = metrics["attr_acc_time"] = 0

    metrics["det_acc"] = correct_det / total_gt if total_gt else 0
    metrics["avg_detections"] = total_detections / len(loader.dataset) if len(loader.dataset) > 0 else 0

    return metrics

# =============================================================================
# Carregar e avaliar cada modelo
# =============================================================================
# Cria um dicionário com os modelos e carrega os pesos (verifique que os arquivos best_{nome}.pth existem)
models = {
    "multiscale": MultiScaleCNN(num_classes),
    "tinyyolo": TinyYOLOStyle(num_classes),
    "sharedtask": SharedTrunkMultiTask(num_classes, num_weather, num_scene, num_time),
    "attention": CNNWithAttention(num_classes),
    "pyramid": PyramidCNN(num_classes),
}

results = {}
for name, model in models.items():
    weight_path = f"best_{name}.pth"
    if os.path.exists(weight_path):
        print(f"✅ Carregando pesos para {name} do arquivo {weight_path}")
        state = torch.load(weight_path, map_location=device)
        model.load_state_dict(state)
        model.to(device)
        print(f"🔧 Avaliando modelo: {name}")
        results[name] = evaluate_model(model, val_loader, device)
    else:
        print(f"⚠️ Peso do modelo {name} não encontrado em {weight_path}. Ignorando.")

# Salvar os resultados de avaliação em um arquivo JSON
os.makedirs("logs", exist_ok=True)
with open("logs/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("📄 Resultados de avaliação salvos em 'logs/eval_results.json'")

# =============================================================================
# Gerar gráfico comparativo das métricas
# =============================================================================
import numpy as np

metrics = ["attr_acc_weather", "attr_acc_scene", "attr_acc_time", "det_acc", "avg_detections"]
labels = list(results.keys())
# Preparar os valores para cada métrica
values = {metric: [results[model_name][metric] for model_name in labels] for metric in metrics}

plt.figure(figsize=(12, 6))
for metric in metrics:
    plt.plot(labels, values[metric], marker='o', label=metric)
plt.title("Comparação de Métricas entre Modelos")
plt.ylabel("Valor")
plt.grid(True)
plt.legend()
plt.savefig("logs/eval_metrics_comparison.png")
print("📊 Gráfico salvo em 'logs/eval_metrics_comparison.png'")
plt.show()
